# Agent continuous evaluation: Aria RM briefing agent

Demonstrates how to set up always-on quality checks for a Foundry agent using the `azure-ai-projects` continuous evaluation SDK, then verify results in the Foundry Portal Monitor tab.

The agent we evaluate is **`aria-rm-briefing-agent`** - the intent-level MCP briefing assistant for Contoso Private Investments relationship managers, created in [08-05b-01-private-banking-agent-setup.ipynb](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb). The continuous-eval pattern itself is domain-agnostic; the private-banking narrative just makes the relevance scoring concrete for compliance and risk attendees.

This lab targets the **admin project on the hub account** (`project-admin-{suffix}`), not a spoke project. Foundry's continuous-eval LLM-as-judge needs a model deployment on the agent's project's parent account - our spoke accounts have zero deployments by design (the deny-model-deployments policy is enforced at every spoke RG). The admin project natively hosts the centrally-managed deployments, so it's the natural home for a centrally-operated evaluation pipeline. Aria also lives on the admin project, so the eval rule and the agent it filters on share the same surface.

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root, then select the `.venv` kernel in VS Code.
2. **Azure CLI logged in**: `az login`. Used to derive the deterministic subscription suffix and locate the admin project.
3. **Hub estate deployed** (project pattern setup): provides `aif-core-{suffix}` with the model deployments and `project-admin-{suffix}`.
4. **Run [`08-05b-01-private-banking-agent-setup.ipynb`](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb) first** - creates `aria-rm-briefing-agent`. This notebook fails fast if the agent is missing.
5. **`.env`**: only `CHAT_MODEL` (set by the core gateway deployment) is required for this lab - needed as the LLM-judge deployment name.
6. **IAM**: the admin project's managed identity must have the **Foundry User** role on the project resource. This is assigned automatically by [08-07-01-deploy-observability-infra.ipynb](08-07-01-deploy-observability-infra.ipynb) - no manual setup needed.

## What you'll learn

| Concept | Description |
|---------|-------------|
| **Eval objects** | OpenAI-compatible eval definitions that describe what to measure |
| **Evaluation rules** | Continuous rules that apply an eval to sampled agent responses automatically |
| **Monitor tab** | Foundry Portal surface where continuous eval results appear |

## References

- [Agent Monitoring Dashboard](https://learn.microsoft.com/azure/ai-foundry/observability/how-to/how-to-monitor-agents-dashboard?view=foundry)
- [08-07-04-real-time-observability.md](08-07-04-real-time-observability.md) - portal navigation and Monitor Settings overview
- [08-07-03-agent-observability.ipynb](08-07-03-agent-observability.ipynb) - companion notebook that traces this same agent via OpenTelemetry

---
## Step 1: Environment setup

In [1]:
import os, subprocess, time, hashlib
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ['CHAT_MODEL']

# Continuous evaluation needs an LLM-as-judge model deployment on the agent's project's
# parent account. Spoke accounts have zero deployments by design (the deny-model-deployments
# policy is enforced at every spoke RG). This lab therefore targets the *admin* project on
# the hub account, which natively hosts the centrally-managed deployments.
SUB_ID = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

# Agent and judge resolve to the same direct deployment on the admin project's parent account
# (no APIM connection prefix needed since the model lives on the same account).
MODEL = CHAT_MODEL  # e.g., 'gpt-4.1-mini'

# Aria is created by 08-05b; we look it up here rather than redefining.
AGENT_NAME = 'aria-rm-briefing-agent'

print('Configuration loaded')
print(f'  Project endpoint: {PROJECT_ENDPOINT}')
print(f'  Model:            {MODEL}')
print(f'  Agent:            {AGENT_NAME}')

Configuration loaded
  Project endpoint: https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
  Model:            gpt-4.1-mini
  Agent:            aria-rm-briefing-agent


---
## Step 2: IAM prerequisite (automatic)

Continuous evaluation requires the Foundry project's **managed identity** to have the **Foundry User** role on the project resource. [`08-07-01-deploy-observability-infra.ipynb`](08-07-01-deploy-observability-infra.ipynb) assigns this automatically when it deploys the admin module - no manual `az role assignment` step needed.

> If the role is missing (e.g. you ran an older version of 08-07-01), the `evaluation_rules.create_or_update` call in Step 5 will return a 403. Re-run 08-07-01 to fix.

---
## Step 3: Reference the Aria agent

We're not creating a new agent - `aria-rm-briefing-agent` was created on the admin project by [08-05b](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb). We look up the latest version and attach the continuous-eval rule to it.

> If the agent doesn't exist, the cell below raises with a pointer back to 08-05b. We deliberately don't auto-create - keeps the dependency explicit.

In [2]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import ResourceNotFoundError

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

try:
    existing_versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
except ResourceNotFoundError:
    existing_versions = []

if not existing_versions:
    raise RuntimeError(
        f"Agent '{AGENT_NAME}' not found on the admin project.\n"
        f"  Run 08-05b-01-private-banking-agent-setup.ipynb first to create it."
    )

agent         = existing_versions[0]
AGENT_VERSION = agent.version

print(f'Found agent: {agent.name} v{AGENT_VERSION}')
print(f'  Model (set in 08-05b): {MODEL}')

Found agent: aria-rm-briefing-agent v1
  Model (set in 08-05b): gpt-4.1-mini


---
## Step 4: Create eval object

An eval object defines what to measure. We use the `builtin.relevance` evaluator - it checks whether the agent's response is relevant to the user's query.

In [3]:
# Foundry's eval endpoints are project-scoped; the project client's built-in OpenAI-compatible
# client targets the project endpoint with Entra auth. The judge model below resolves directly
# to a deployment on the admin project's parent account - no gateway hop needed.
openai_client = project_client.get_openai_client()

eval_object = openai_client.evals.create(
    name='Continuous Evaluation - Relevance (Aria RM Briefing)',
    data_source_config={
        'type': 'azure_ai_source',
        'scenario': 'responses'
    },
    testing_criteria=[
        {
            'type': 'azure_ai_evaluator',
            'name': 'relevance_check',
            'evaluator_name': 'builtin.relevance',
            'data_mapping': {
                'query': '{{item.query}}',
                'response': '{{item.response}}',
            },
            'initialization_parameters': {
                'deployment_name': MODEL,
            },
        }
    ],
)

print(f'Eval object created')
print(f'  ID:   {eval_object.id}')
print(f'  Name: {eval_object.name}')

Eval object created
  ID:   eval_aaa6df72ee5842a7b754227023339ae0
  Name: Continuous Evaluation - Relevance (Aria RM Briefing)


---
## Step 5: Create continuous evaluation rule

An evaluation rule attaches an eval object to an agent. Foundry samples agent responses and runs the eval automatically - no per-request code changes needed.

In [4]:
from azure.ai.projects.models import (
    EvaluationRule,
    ContinuousEvaluationRuleAction,
    EvaluationRuleFilter,
    EvaluationRuleEventType,
)

rule = project_client.evaluation_rules.create_or_update(
    id='continuous-relevance-rule-aria',
    evaluation_rule=EvaluationRule(
        display_name='Continuous Relevance Evaluation (Aria RM Briefing)',
        action=ContinuousEvaluationRuleAction(
            eval_id=eval_object.id,
            max_hourly_runs=100
        ),
        event_type=EvaluationRuleEventType.RESPONSE_COMPLETED,
        filter=EvaluationRuleFilter(agent_name=agent.name),
        enabled=True,
    ),
)

print(f'Evaluation rule created')
print(f'  ID:           {rule.id}')
print(f'  Display name: {rule.display_name}')
print(f'  Enabled:      {rule.enabled}')

Evaluation rule created
  ID:           continuous-relevance-rule-aria
  Display name: Continuous Relevance Evaluation (Aria RM Briefing)
  Enabled:      True


---
## Step 6: Generate agent traffic

Send a few queries through the agent to trigger the continuous evaluation rule. Foundry samples these responses and runs the relevance evaluator.

In [5]:
# Use the OpenAI-compatible Responses API via the project client
responses_client = project_client.get_openai_client()

# Five RM-briefing queries exercising five different intent-level MCP tools.
# Real client IDs (cli-001 .. cli-005) from assets/contoso-private-banking-dataset/.
queries = [
    'Prepare a briefing for my meeting with the Eichmann Foundation (cli-002).',
    'Walk me through the portfolio drift for the Lindemann Family Office (cli-003).',
    'What recent activity has occurred for the Berger Family Trust (cli-001)?',
    'Find research on private equity and concentrated-position legacy holdings.',
    'Give me the client context for the Riedi Pension Plan (cli-005).',
]

print(f'Sending {len(queries)} queries through {AGENT_NAME}...\n')

for query in queries:
    print(f'User: {query}')
    start = time.time()
    response = responses_client.responses.create(
        input=query,
        extra_body={
            'agent_reference': {
                'name': AGENT_NAME,
                'version': AGENT_VERSION,
                'type': 'agent_reference'
            }
        },
    )
    duration = (time.time() - start) * 1000
    output_text = getattr(response, 'output_text', None) or str(response.output)
    print(f'Aria: {str(output_text)[:200]}')
    print(f'  Duration: {duration:.0f}ms\n')
    time.sleep(1)

print('Agent traffic complete - continuous evaluation rule will process sampled responses.')

Sending 5 queries through aria-rm-briefing-agent...

User: Prepare a briefing for my meeting with the Eichmann Foundation (cli-002).
Aria: Here is the briefing for your meeting with the Eichmann Foundation (Philanthropic Foundation, RM Stefan Hofer):

- Portfolio AUM: CHF 48,461,182 as of 2026-05-10
- Asset allocation: Equities 26.12%, F
  Duration: 18120ms

User: Walk me through the portfolio drift for the Lindemann Family Office (cli-003).
Aria: The portfolio drift analysis for the Lindemann Family Office (cli-003) shows the following key points:

Asset Class Drift:
- Fixed Income is underweight versus the target by 6.75 percentage points (ta
  Duration: 20722ms

User: What recent activity has occurred for the Berger Family Trust (cli-001)?
Aria: Over the past 30 days, the Berger Family Trust has had 2 transactions:

1. On May 2, 2026, a buy transaction of 3,000 units at a price of 210.4 USD each, amounting to CHF 562,000, categorized as a the
  Duration: 13927ms

User: Find research

---
## Step 7: List active evaluation rules

Confirm the rule is registered and active before checking the portal.

In [6]:
rules = list(project_client.evaluation_rules.list())

print(f'Active evaluation rules ({len(rules)} total):\n')
for r in rules:
    status = 'enabled' if r.enabled else 'disabled'
    print(f'  {r.id:<40} {r.display_name:<45} [{status}]')

# Derive portal URL from endpoint
# Format: https://aif-spoke-multi-{suffix}.services.ai.azure.com/api/projects/{project}
# Portal URL: https://ai.azure.com/
print()
print('To view evaluation results in the portal:')
print('  1. Open https://ai.azure.com')
print('  2. Navigate to Build > your agent > Monitor tab')
print('  Results may take a few minutes to appear after the first traffic batch.')

Active evaluation rules (1 total):

  continuous-relevance-rule-aria           Continuous Relevance Evaluation (Aria RM Briefing) [enabled]

To view evaluation results in the portal:
  1. Open https://ai.azure.com
  2. Navigate to Build > your agent > Monitor tab
  Results may take a few minutes to appear after the first traffic batch.


---
## Step 8: Verify in the Foundry portal

After generating traffic, verify continuous evaluation results in the portal.

### Navigation

1. Open `https://ai.azure.com` - confirm **New Foundry** toggle is on
2. Go to **Build** page → select `aria-rm-briefing-agent`
3. Select the **Monitor** tab

### What to look for

- **Evaluation metrics** panel: shows relevance scores for sampled responses
- **Token usage** and **Run success rate** charts should reflect the 5 queries sent in Step 6
- Under **Monitor Settings**, the `continuous-relevance-rule-aria` rule should appear as **Enabled**

> Results may take up to a few minutes to appear after the first traffic batch. If the Evaluation metrics panel shows no data, wait 5 minutes and refresh.

For a full description of each metric and Monitor Settings feature, see [08-07-04-real-time-observability.md](08-07-04-real-time-observability.md).

---
## Step 9: Cleanup

The `aria-rm-briefing-agent` is owned by 08-05b - **do not delete it here**. Cleanup is scoped to the evaluation rule and eval object this notebook created.

In [7]:
# Do NOT delete the Aria agent - it is owned by 08-05b and reused by 08-07-03.
# Only delete this lab's evaluation rule and eval object if you want a clean slate:
#
# project_client.evaluation_rules.delete('continuous-relevance-rule-aria')
# print('Deleted evaluation rule: continuous-relevance-rule-aria')
#
# openai_client.evals.delete(eval_object.id)
# print(f'Deleted eval object: {eval_object.id}')